In [ ]:
import polars as pl
from datetime import date
from pathlib import Path
from common import daterange
import json
import yaml

with open("../config.yaml", "r") as fp:
    config = yaml.safe_load(fp)

days = []
for i in config["days"]:
    days += list(daterange(i["start"], i["end"]))

In [ ]:
DIR = f"{config['dirs']['output']}/partitioned"

In [ ]:
missing = []
for i in range(256):
    data = pl.DataFrame()
    for d in days:
        if not Path(f"{DIR}/{i:02x}/pings_{d.strftime('%Y_%m_%d')}.parquet").is_file():
            missing.append((i, d))
            continue
        dd = pl.read_parquet(f"{DIR}/{i:02x}/pings_{d.strftime('%Y_%m_%d')}.parquet")
        data = pl.concat([data, dd])
        del dd
    data.sort(["device_id", "timestamp"]).write_parquet(f"{DIR}/{i:02x}/pings.parquet")
    del data

with open("../output/debug/03_merge_partitions_missing_log.json", "w") as fp:
    json.dump([(i[0], i[1].strftime("%Y-%m-%d")) for i in missing], fp, indent=1)

# runs 45m

In [ ]:
df = pl.read_parquet(f"{DIR}/00/pings.parquet")
df.write_csv(f"{DIR}/00/pings.csv")